In [9]:
from typesafe_sdk import AsyncTypeSafeClient, Choice, Noul, Score

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import pandas as pd

from pathlib import Path

from PIL import Image, ImageDraw
from typesafe_sdk import Box, Image as TSImage, TypeSafeClient

In [10]:
SOURCE = Path("cars2.jpg")          # your input image
OUTPUT = Path("cars-boxed.jpg")    # where to save the result
TILE_DIR = Path("tiles")           # where the tile crops are written
TARGET = "red car"                 # describe the object to box
GRID = 3                           # split the image into GRID x GRID tiles
OVERLAP = 0.15                     # extra overlap between neighbouring tiles, as a fraction of the image size
PAD = 0.0                          # optional padding, as a fraction of the image size
MIN_CONFIDENCE = 0.3               # ignore boxes the model is unsure about
NOUL_THRESHOLD = 0.3               # presence score (0..1) needed to keep a tile's box
IOU_THRESHOLD = 0.5                # boxes overlapping more than this are the same object

image = Image.open(SOURCE).convert("RGB")
width, height = image.size

# Box returns one box per question and always returns a box, so cover the image
# with overlapping tiles and ask a Box plus a presence Noul on each tile.
TILE_DIR.mkdir(exist_ok=True)
tiles = {}
step = 1 / GRID
for row in range(GRID):
    for col in range(GRID):
        left = max(0.0, col * step - OVERLAP / 2)
        top = max(0.0, row * step - OVERLAP / 2)
        right = min(1.0, (col + 1) * step + OVERLAP / 2)
        bottom = min(1.0, (row + 1) * step + OVERLAP / 2)
        name = f"tile_{row}_{col}"
        path = TILE_DIR / f"{name}.jpg"
        image.crop((
            round(left * width),
            round(top * height),
            round(right * width),
            round(bottom * height),
        )).save(path, "JPEG", quality=92)
        tiles[name] = (left, top, right, bottom, path)

# One request per tile: a single-image state keeps each Noul and Box question
# pointed at the image it was written for
answers = {}
with TypeSafeClient() as client:
    for name, tile in tiles.items():
        result = client.system_one(
            model="jev-preview",       # Box questions need Jev 1.14
            state={"tile": TSImage(tile[4])},
            questions={
                "present": Noul(instructions=f"Is there a {TARGET} in `tile`?"),
                "box": Box(instructions=f"the {TARGET} in `tile`"),
            },
            extra_headers={"TypeSafe-Danger-Zone-Images": "true"},
        )
        answers[name] = result.answers

print(f"Asked {len(answers)} tiles")

Asked 9 tiles


In [11]:
# Whole-image count check: none, one, or several red cars
# (uses TARGET and SOURCE from the config cell above)
with TypeSafeClient() as client:
    count_result = client.system_one(
        model="jev-preview",
        state={"photo": TSImage(SOURCE)},
        questions={
            "count": Choice(
                instructions=f"Are there none, one, or several {TARGET}s in `photo`?",
                criteria={
                    "none": f"No {TARGET}s are visible in `photo`",
                    "one": f"Exactly one {TARGET} is visible in `photo`",
                    "several": f"Two or more {TARGET}s are visible in `photo`",
                },
            ),
        },
        extra_headers={"TypeSafe-Danger-Zone-Images": "true"},
    )

count = count_result.answers["count"]
print(f"Count: {count.choice} (confidence {count.confidence:.2f})")
print({option: round(p, 2) for option, p in count.probabilities.items()})

Count: several (confidence 0.43)
{'none': 0.0, 'one': 0.38, 'several': 0.62}


In [12]:
def tile_to_image(edges, tile):
    # Convert edges from fractions of a tile to fractions of the full image
    left, top, right, bottom, _ = tile
    tile_w, tile_h = right - left, bottom - top
    return (
        left + edges.left * tile_w,
        top + edges.top * tile_h,
        left + edges.right * tile_w,
        top + edges.bottom * tile_h,
    )


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


candidates = []
for name, tile in tiles.items():
    presence = answers[name]["present"]
    box_answer = answers[name]["box"]
    if presence.noul is None:
        # Fail loudly instead of silently treating a missing value as "no"
        raise ValueError(f"{name}: Noul value is None, answer was {presence!r}")
    print(f"{name}: noul={presence.noul:.2f} confidence={box_answer.confidence:.2f}")
    if presence.noul < NOUL_THRESHOLD or box_answer.confidence < MIN_CONFIDENCE:
        continue
    candidates.append((tile_to_image(box_answer.box, tile), box_answer.confidence))

# Tiles overlap, so the same car can be found more than once: keep the most confident one
cars = []
for box, confidence in sorted(candidates, key=lambda c: c[1], reverse=True):
    if all(iou(box, kept) < IOU_THRESHOLD for kept, _ in cars):
        cars.append((box, confidence))

print(f"Found {len(cars)} {TARGET}(s)")

tile_0_0: noul=0.14 confidence=0.07
tile_0_1: noul=0.15 confidence=0.06
tile_0_2: noul=0.51 confidence=0.38
tile_1_0: noul=0.23 confidence=0.04
tile_1_1: noul=0.08 confidence=0.22
tile_1_2: noul=0.48 confidence=0.19
tile_2_0: noul=0.96 confidence=0.56
tile_2_1: noul=0.90 confidence=0.27
tile_2_2: noul=0.33 confidence=0.87
Found 3 red car(s)


In [13]:
# Draw every box on the full image
draw = ImageDraw.Draw(image)
line_width = max(2, round(min(width, height) / 200))
for (left, top, right, bottom), confidence in cars:
    left = max(0.0, left - PAD)
    top = max(0.0, top - PAD)
    right = min(1.0, right + PAD)
    bottom = min(1.0, bottom + PAD)
    box_px = (
        round(left * width),
        round(top * height),
        round(right * width),
        round(bottom * height),
    )
    draw.rectangle(box_px, outline=(255, 0, 0), width=line_width)
    print(f"{box_px} confidence={confidence:.2f}")

image.save(OUTPUT, "JPEG", quality=92)
print(f"Saved {OUTPUT} with {len(cars)} box(es)")

(774, 570, 843, 828) confidence=0.87
(175, 687, 361, 867) confidence=0.56
(1077, 172, 1199, 289) confidence=0.38
Saved cars-boxed.jpg with 3 box(es)
